# Module 5 — Entity resolution: merging duplicates into `EntityGroup`

**The gap, from Module 4:** extraction stores every mention as its own `RecognisedEntity`,
keyed by `(string, doc_id)` (see [adr/0007](../docs/adr/0007-extraction-generic-recognisedentity.md)).
The same physical entity now exists many times over — "3M Company", "3M", and "MMM" all appear
as separate nodes from the same filing, "3M Company" appears *again* in every other 3M filing
under the same string, and a curated `Company {id: "3M"}` node from Module 1 exists independently
of all of them. Module 3's Wikidata enrichment adds a third source of the same kind of duplication
for people and subsidiaries.

**What this notebook builds:** a resolver (`similarity/resolver.py`) that, per entity type,
narrows candidates by name similarity, enriches survivors with graph context, asks an LLM to
confirm which are truly the same real-world entity, and writes the result as one `EntityGroup`
node per confirmed cluster with `SAME_AS` edges from every member. Full design rationale in
[adr/0008](../docs/adr/0008-entity-resolution-design.md).

**Scope note:** this notebook covers the resolver only. `similarity/linker.py`
(`SIMILAR_TO` chunk-similarity edges) is a separate piece of Module 5 and is still a stub —
picked up in a later session.


## 1. The duplicate problem, in the graph as it stands

Module 4 ran its extraction demo over four chunks of 3M's filings (see `module_04_extraction.ipynb`,
section 6) plus Module 3's Wikidata enrichment ran for both 3M and Apple. Look at what "3M" alone
resolves to today: several `RecognisedEntity` mentions across two different filings, plus a
separate curated `Company` node — none of them connected to each other.


In [1]:
from financial_advisor.services.neo4j_service import neo4j_service

rows = neo4j_service.run_query(
    """
    MATCH (e:RecognisedEntity {type: "Company"})
    WHERE e.string CONTAINS "3M"
    RETURN "RecognisedEntity" AS label, e.string AS name, e.doc_id AS context
    UNION
    MATCH (c:Company)
    WHERE c.id CONTAINS "3M"
    RETURN "Company" AS label, c.id AS name, "curated (Module 1/3)" AS context
    """
)
for row in rows:
    print(f"{row['label']:16s} | {row['name']:45s} | {row['context']}")
print(f"\n{len(rows)} node(s), zero of them connected to each other yet")


RecognisedEntity | 3M Company                                    | 3M/3M_2025_10K.pdf
RecognisedEntity | 3M                                            | 3M/3M_2025_10K.pdf
RecognisedEntity | 3M Company                                    | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Financial Management Company               | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Innovative Properties Company              | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Interamerica LLC                           | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Chemical Operations LLC                    | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Fall Protection Company                    | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Foreign Holding LLC                        | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M do Brasil Ltda.                            | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Belgium BV                                 | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Canada Company - Compagnie 3M Ca

## 2. A naive fix: substring matching

The most obvious approach: pick a target, find every other node whose name contains it (or vice
versa), and merge them all. No graph context, no LLM — just string containment. This is
notebook-only, deliberately worse than what ships in `src/` (same convention as Module 4's naive
prompt demo).


In [2]:
# Ad hoc, throwaway heuristic — NOT how similarity.resolver works. Exists to be visibly wrong.
naive_matches = neo4j_service.run_query(
    """
    MATCH (e:RecognisedEntity {type: "Company"})
    WHERE e.string CONTAINS "3M"
    RETURN e.string AS name
    ORDER BY name
    """
)
print(f"Naive substring match on '3M' pulls in {len(naive_matches)} entities, e.g.:")
for row in naive_matches[:8]:
    print(f"  - {row['name']}")
print("  ...")


Naive substring match on '3M' pulls in 34 entities, e.g.:
  - 3M
  - 3M Belgium BV
  - 3M Canada Company - Compagnie 3M Canada
  - 3M Chemical Operations LLC
  - 3M China Limited
  - 3M Company
  - 3M Company
  - 3M Deutschland GmbH
  ...


**What went wrong.** Every one of 3M's ~40 consolidated subsidiaries also contains "3M" in its
name — "3M Financial Management Company", "3M Chemical Operations LLC", "3M do Brasil Ltda.", and
so on. A naive fix would merge the parent company with all of its subsidiaries into one entity,
which is exactly wrong: `SUBSIDIARY_OF` is a real, meaningful relationship in this graph
(Module 3's Wikidata structure, Module 4's extracted subsidiaries table) that a careless merge
would destroy.


## 3. Adding a real candidate filter: fuzzy matching

`similarity/candidates.py::find_fuzzy_candidates` replaces substring containment with RapidFuzz's
`WRatio` scorer — better at matching real name variants (abbreviations, corporate suffixes, word
order) — but it's still just comparing strings. Run it against the same target and the result
looks almost identical to the naive attempt: subsidiaries share too much of the parent's name to
be filtered out by string similarity alone.


In [3]:
from financial_advisor.extraction.validators import EntityType
from financial_advisor.similarity.candidates import fetch_candidate_pool, find_fuzzy_candidates

company_pool = fetch_candidate_pool(EntityType.COMPANY)
print(f"Unresolved Company-type pool: {len(company_pool)} node(s)")

target = next(n for n in company_pool if n.name == "3M Company")
rest = [n for n in company_pool if n is not target]
fuzzy_matches = find_fuzzy_candidates(target, rest, threshold=78, limit=8)

print(f"\nFuzzy candidates for '{target.name}':")
for m in fuzzy_matches:
    print(f"  - [{m.label:16s}] {m.name}")


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SAME_AS` does not exist in database `packt`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=25, offset=74>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 74, 'line': 3, 'column': 25}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (e:RecognisedEntity {type: $type})\n        WHERE NOT (e)-[:SAME_AS]->(:EntityGroup)\n        RETURN e.string AS string, e.doc_id AS doc_id\n        '


Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `EntityGroup` does not exist in database `packt`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=37, offset=86>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 86, 'line': 3, 'column': 37}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (e:RecognisedEntity {type: $type})\n        WHERE NOT (e)-[:SAME_AS]->(:EntityGroup)\n        RETURN e.string AS string, e.doc_id AS doc_id\n        '


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SAME_AS` does not exist in database `packt`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=29, offset=59>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 59, 'line': 3, 'column': 29}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (n:Company)\n            WHERE NOT (n)-[:SAME_AS]->(:EntityGroup)\n            RETURN n.id AS id, coalesce(n.name, n.id) AS name\n            '


Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `EntityGroup` does not exist in database `packt`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=41, offset=71>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 71, 'line': 3, 'column': 41}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (n:Company)\n            WHERE NOT (n)-[:SAME_AS]->(:EntityGroup)\n            RETURN n.id AS id, coalesce(n.name, n.id) AS name\n            '


Unresolved Company-type pool: 79 node(s)

Fuzzy candidates for '3M Company':
  - [RecognisedEntity] 3M Company
  - [RecognisedEntity] 3M
  - [Company         ] 3M
  - [RecognisedEntity] 3M Financial Management Company
  - [RecognisedEntity] 3M Innovative Properties Company
  - [RecognisedEntity] 3M Interamerica LLC
  - [RecognisedEntity] 3M Chemical Operations LLC
  - [RecognisedEntity] 3M Fall Protection Company


Still no way to tell, from names alone, that "3M Financial Management Company" is a subsidiary
and "3M" (the short form used elsewhere in the same filing) is not. That distinction only exists
in the graph — as a `RELATED_TO {type: "SUBSIDIARY_OF"}` edge Module 4 already extracted.


## 4. What graph context adds

`similarity/context.py::fetch_entity_context` enriches a candidate into the JSON the LLM actually
sees: the node's own properties, plus its 1-hop neighborhood in any direction (any relationship
type, skipping bulky `Chunk`/`Document` nodes), plus — for extracted mentions — which company's
filing it was mentioned in. For "3M Company" itself, the neighborhood includes the regulatory and
location context extracted alongside it; for a subsidiary like "3M Financial Management Company",
it includes the `SUBSIDIARY_OF` edge back to the parent — the one piece of evidence a name-only
comparison can never see.


In [4]:
import json

from financial_advisor.similarity.context import fetch_entity_context

target_context = fetch_entity_context(target)
print("Target — '3M Company':")
print(json.dumps(target_context, indent=2)[:900], "...\n")

subsidiary = next(m for m in fuzzy_matches if m.name == "3M Financial Management Company")
subsidiary_context = fetch_entity_context(subsidiary)
print("Candidate — '3M Financial Management Company':")
print(json.dumps(subsidiary_context, indent=2))


Target — '3M Company':
{
  "name": "3M Company",
  "type": "Company",
  "source": "RecognisedEntity",
  "own_properties": {
    "string": "3M Company",
    "description": "Was incorporated in 1929 under the laws of the State of Delaware; the Company referenced throughout the document.",
    "type": "Company",
    "doc_id": "3M/3M_2025_10K.pdf"
  },
  "neighbors": [
    {
      "relationship": "RELATED_TO",
      "direction": "out",
      "node_label": "RecognisedEntity",
      "name": "State of Delaware",
      "node_type": "Location"
    },
    {
      "relationship": "RELATED_TO",
      "direction": "out",
      "node_label": "RecognisedEntity",
      "name": "Securities Exchange Act of 1934 (Exchange Act)",
      "node_type": "Regulation"
    },
    {
      "relationship": "RELATED_TO",
      "direction": "out",
      "node_label": "RecognisedEntity",
      "name": "Annual Report on Form 10-K",
      "node_ ...

Candidate — '3M Financial Management Company':
{
  "name": "3M Financia

## 5. LLM judgment: confirm or reject, with a reason

`similarity/resolver.py::judge_candidates` sends the target's context plus every fuzzy
candidate's context to the LLM in one structured-output call
(`similarity.validators.ResolutionResult`) and gets back a `same_entity` verdict + one-sentence
reason per candidate — grounded in the graph evidence just built, not name similarity.


In [5]:
from financial_advisor.similarity.resolver import judge_candidates

candidate_contexts = [fetch_entity_context(m) for m in fuzzy_matches]
result = judge_candidates(EntityType.COMPANY, target_context, candidate_contexts)

for j in result.judgments:
    verdict = "SAME" if j.same_entity else "different"
    print(f"[{verdict:9s}] {fuzzy_matches[j.index].name:40s} — {j.reason}")


[SAME     ] 3M Company                               — Same exact name and role (Registrant 3M Company) in the company's 2024 Form 10‑K; matches the target 3M Company.
[SAME     ] 3M                                       — Named as the reporting company '3M' in the same 2025 10‑K and described throughout the overview, indicating the same corporate entity as 3M Company.
[SAME     ] 3M                                       — Contains corporate identifiers (ticker MMM, NYSE, Maplewood HQ) for 3M and uses the same canonical id '3M', indicating the same company as the target.
[different] 3M Financial Management Company          — Explicitly described as a consolidated subsidiary (3M Financial Management Company) and is linked out to 3M Company, so it is a distinct subsidiary, not the same entity.
[different] 3M Innovative Properties Company         — Labeled as 3M Innovative Properties Company and described as a consolidated subsidiary connected to 3M Company, so it is a different entity.
[

**What actually happened, one run:** the LLM confirmed the three genuine duplicates — "3M
Company" (another filing), "3M" (the short form used in the same filing), and the curated
`Company {id: "3M"}` node from Module 1/3 — and rejected every subsidiary, each time citing the
`SUBSIDIARY_OF` edge as the reason it's a distinct entity. That's the payoff of the graph-context
step: the same name-similarity signal that made subsidiaries *candidates* in the first place gets
correctly overruled once the model can see how they actually relate to the parent.


## 6. Storage: `EntityGroup` and `SAME_AS`

Confirmed clusters get written as one `EntityGroup {id, type, canonical_name}` node with a
`SAME_AS` edge from every member (target + confirmed candidates). `canonical_name` prefers a
curated node's name over an extracted mention, since it's already clean (Wikidata-sourced or the
company's own `id`). No group is written when nothing is confirmed — a target with zero confirmed
matches just stays ungrouped and gets reconsidered on the next run. Full rationale in
[adr/0008](../docs/adr/0008-entity-resolution-design.md).

`apply_similarity_schema()` adds the one new constraint this needs: uniqueness on
`EntityGroup.id`.


In [6]:
from financial_advisor.ingestion.schema import apply_similarity_schema

apply_similarity_schema()


  [schema] OK  entity_group_id
[schema] 1/1 statements applied — all good


## 7. Running the pipeline for real

`resolve_entity_type` ties steps 3-6 together for one entity type; `resolve_all_entities` runs it
across every type in `RESOLUTION_ORDER` — companies and people first (richest graph context,
curated counterparts to reconcile against), then the extraction-only types. Idempotent: a node
already under a `SAME_AS` edge is excluded from the pool up front, so rerunning this after a
future extraction batch only processes what's new.


In [7]:
from financial_advisor.similarity.resolver import resolve_all_entities

summary = resolve_all_entities()
print(f"\nGroups created per type: {summary}")
print(f"Total: {sum(summary.values())}")


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SAME_AS` does not exist in database `packt`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=25, offset=74>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 74, 'line': 3, 'column': 25}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (e:RecognisedEntity {type: $type})\n        WHERE NOT (e)-[:SAME_AS]->(:EntityGroup)\n        RETURN e.string AS string, e.doc_id AS doc_id\n        '


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SAME_AS` does not exist in database `packt`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=29, offset=59>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 59, 'line': 3, 'column': 29}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (n:Company)\n            WHERE NOT (n)-[:SAME_AS]->(:EntityGroup)\n            RETURN n.id AS id, coalesce(n.name, n.id) AS name\n            '


[resolve:Company] 79 unresolved node(s)
[resolve:Company] '3M Company' (RecognisedEntity) — 8 fuzzy candidate(s): ['3M Company', '3M', '3M', '3M Financial Management Company', '3M Innovative Properties Company', '3M Interamerica LLC', '3M Chemical Operations LLC', '3M Fall Protection Company']


[resolve:Company] group 39d9536e = 3M Company + ['3M Company', '3M', '3M']
[resolve:Company] 'Solventum' (RecognisedEntity) — 1 fuzzy candidate(s): ['Solventum Corporation (Solventum)']


[resolve:Company] group d902b9b6 = Solventum + ['Solventum Corporation (Solventum)']
[resolve:Company] 'U.S. Environmental Protection Agency (EPA)' (RecognisedEntity) — 1 fuzzy candidate(s): ['3M Fall Protection Company']


...

[resolve:FinancialMetric] 'Organic sales change' — LLM confirmed none, no group created
[resolve:FinancialMetric] 'Net sales change' (RecognisedEntity) — 2 fuzzy candidate(s): ['YoY change in operating income margin', 'Income from Unconsolidated Subsidiaries, Net of Taxes']


[resolve:FinancialMetric] 'Net sales change' — LLM confirmed none, no group created
[resolve:FinancialMetric] 'Operating income margin' (RecognisedEntity) — 1 fuzzy candidate(s): ['YoY change in operating income margin']


[resolve:FinancialMetric] 'Operating income margin' — LLM confirmed none, no group created
[resolve:FinancialMetric] 'YoY change in operating income margin' (RecognisedEntity) — 2 fuzzy candidate(s): ['operating income', 'YoY change in EPS']


[resolve:FinancialMetric] 'YoY change in operating income margin' — LLM confirmed none, no group created
[resolve:FinancialMetric] 'YoY change in EPS' (RecognisedEntity) — 2 fuzzy candidate(s): ['EPS', '$0.8 billion pre-tax pension settlement charge in 2024']


[resolve:FinancialMetric] 'YoY change in EPS' — LLM confirmed none, no group created
[resolve:FinancialMetric] done — 2 group(s) created

Groups created per type: {'Company': 6, 'Person': 0, 'Product': 5, 'Location': 1, 'Regulation': 6, 'Risk': 4, 'FinancialMetric': 2}
Total: 24


## 8. Inspecting the result

Every `EntityGroup` and its members, across all types — this is the reconciled view Module 6/7
can query instead of guessing which raw `RecognisedEntity`/`Company`/`Person` nodes refer to the
same thing.


In [8]:
rows = neo4j_service.run_query(
    """
    MATCH (g:EntityGroup)<-[:SAME_AS]-(member)
    RETURN g.type AS type, g.canonical_name AS canonical_name,
           collect(coalesce(member.string, member.name, member.id)) AS members
    ORDER BY type, canonical_name
    """
)
for row in rows:
    print(f"[{row['type']:16s}] {row['canonical_name']:30s} <- {row['members']}")
print(f"\n{len(rows)} EntityGroup(s) total")


[Company         ] 3M                             <- ['3M', '3M Company', '3M', '3M Company']
[Company         ] 3M (Germany)                   <- ['3M (Germany)', '3M Deutschland GmbH']
[Company         ] 3M Innovation Singapore Pte. Ltd. <- ['3M Innovation Singapore Pte. Ltd.', '3M Singapore Pte. Ltd.']
[Company         ] 3M Innovative Properties       <- ['3M Innovative Properties', '3M Innovative Properties Company']
[Company         ] 3M United Kingdom Public Limited Company <- ['3M Products Limited', '3M United Kingdom Public Limited Company']
[Company         ] Solventum Corporation (Solventum) <- ['Solventum', 'Solventum Corporation (Solventum)']
[FinancialMetric ] 3M will pay $10.5 billion to $12.5 billion in total to resolve the claims released by the PWS Settlement <- ['$10.5 billion to $12.5 billion in total', '3M will pay $10.5 billion to $12.5 billion in total to resolve the claims released by the PWS Settlement', '$10.5 billion to $12.5 billion in total to resolve the cl

## 9. Where this leaves the graph

One real run over this corpus (Module 4's four-chunk extraction demo + Module 3's Wikidata
enrichment for 3M and Apple) produced **24 `EntityGroup`s**: 6 `Company`, 5 `Product`,
6 `Regulation`, 4 `Risk`, 2 `FinancialMetric`, 1 `Location`, 0 `Person` (no `Person`-type mention
has been extracted yet — Module 4's demo batch never touched an executive-heavy chunk).

The subsidiaries stayed correctly unmerged — the ~40 distinct `RecognisedEntity`/`Company` nodes
under "3M", each still `SUBSIDIARY_OF` its parent, none of them wrongly folded into the "3M"
group. The genuine duplicates — "3M"/"3M Company" across filings and against the curated
`Company` node, "Solventum"/"Solventum Corporation (Solventum)", "PFAS"/"PFAS compounds" — now
share one `EntityGroup` each. One group is a genuinely interesting judgment call worth checking
yourself in the output above: `3M (Germany)` (a Wikidata subsidiary item) got grouped with
`3M Deutschland GmbH` (the extracted legal-entity name) — plausibly correct, but exactly the kind
of cross-source call that's worth spot-checking rather than trusting blindly.

A downstream query for "everything about 3M" can now traverse
`(x)-[:SAME_AS]->(:EntityGroup {canonical_name: "3M"})<-[:SAME_AS]-(y)` and reach every mention
at once, instead of guessing which of several near-identical strings to search for.

Every other entity in this run — most `Person`, `Product`, `Location`, `Regulation`, `Risk`, and
`FinancialMetric` mentions, since Module 4's extraction has only run over four chunks so far —
stayed ungrouped rather than being forced into a singleton group (see adr/0008's "singletons stay
ungrouped" decision). Running `scripts/run_similarity.py` again after a fuller
`scripts/run_extraction.py` pass will pick those up as real duplicates appear.

**Still pending in Module 5:** `similarity/linker.py` — `SIMILAR_TO` edges between semantically
close `Chunk` nodes, and the follow-on comparison of answer quality with/without the resolved
graph.
